# FAISS Vector Retrieval Notebook

This notebook is prepared for running retrieval after `index.faiss` and `payloads.jsonl` finish downloading.

Expected index directory layout:

```text
data/faiss_index/
  index.faiss
  payloads.jsonl
  id_map.json        # optional, but recommended if available
```

Run the cells from top to bottom. If downloads are not finished yet, the preflight cell will tell you what is still missing.

## 1. Environment setup

In [1]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
%pip install -q faiss-gpu sentence-transformers pandas


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [2]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Useful if the notebook is launched from notebooks/.
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Project root:', PROJECT_ROOT)
print('src on path:', SRC_DIR.exists())


Project root: d:\Uni_Project\Text_Mining\Project
src on path: True


## 2. Configure artifact paths and retrieval settings

Change `INDEX_DIR` if you download the FAISS files somewhere else.

In [3]:
# Directory containing index.faiss + payloads.jsonl (+ optional id_map.json)
INDEX_DIR = PROJECT_ROOT / 'data' / 'faiss_index'

# Must match the embedding model used to build index.faiss.
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'

TOP_K = 30                 # candidates pulled from FAISS before reranking/dedup
TOP_N = 10                 # final chunks returned
SCORE_THRESHOLD = 0.30
EXPAND_UNITS = True
FILTER_PROFILE = 'broad'   # current_law | broad | historical

INDEX_DIR


WindowsPath('d:/Uni_Project/Text_Mining/Project/data/faiss_index')

## 3. Preflight: wait until downloads are complete

In [4]:
required_files = [INDEX_DIR / 'index.faiss', INDEX_DIR / 'payloads.jsonl']
optional_files = [INDEX_DIR / 'id_map.json']

missing = [p for p in required_files if not p.exists()]
if missing:
    print('Downloads are not ready yet. Missing:')
    for p in missing:
        print(' -', p)
else:
    print('Required files found.')
    for p in required_files + optional_files:
        if p.exists():
            print(f'{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB')
        else:
            print(f'{p.name}: not found (optional)')


Required files found.
index.faiss: 5911.63 MB
payloads.jsonl: 4836.18 MB
id_map.json: 60.91 MB


## 4. Load the FAISS store and build retriever

In [ ]:
if missing:
    raise FileNotFoundError('Download index.faiss and payloads.jsonl before running this cell.')

from retrieval.config import VectorIndexConfig
from retrieval.embeddings import SentenceTransformerEmbedder
from retrieval.faiss_store import FaissVectorStore
from retrieval.retriever import VectorRetriever

config = VectorIndexConfig(
    embedding_model=EMBEDDING_MODEL,
    top_k=TOP_K,
    top_n=TOP_N,
    score_threshold=SCORE_THRESHOLD,
    expand_units=EXPAND_UNITS,
)

store = FaissVectorStore.load(INDEX_DIR)
embedder = SentenceTransformerEmbedder(
    EMBEDDING_MODEL,
    query_prefix=config.query_prefix,
    passage_prefix=config.passage_prefix,
)
retriever = VectorRetriever(config=config, embedder=embedder, store=store)

print(f'Loaded FAISS vectors: {store.total_vectors:,}')
print(f'Loaded payloads: {len(store.payloads):,}')
print(f'Embedding dimension: {embedder.dimension}')


## 5. Retrieval helper

In [ ]:
def search(query: str, top_n: int = TOP_N, filter_profile: str = FILTER_PROFILE, score_threshold: float | None = SCORE_THRESHOLD):
    result = retriever.retrieve(
        query,
        filter_profile=filter_profile,
        top_n=top_n,
        score_threshold=score_threshold,
    )
    rows = []
    for rank, chunk in enumerate(result.chunks, start=1):
        rows.append({
            'rank': rank,
            'chunk_id': chunk.chunk_id,
            'citation': chunk.citation_anchor or chunk.citation_label,
            'title': chunk.title,
            'unit_type': chunk.unit_type,
            'validity_group': chunk.validity_group,
            'vector_score': round(chunk.vector_score, 4),
            'rerank_score': round(chunk.rerank_score, 4),
            'text': chunk.chunk_text[:700],
        })
    return rows, result

def show_results(rows):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


## 6. Run a query

In [ ]:
query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'
rows, result = search(query, top_n=10, filter_profile='broad')
print('Filter profile used:', result.filter_profile_used)
print('Total candidates:', result.total_candidates)
print('Empty filter warning:', result.empty_filter_warning)
show_results(rows)


## 7. Optional: inspect one full chunk

In [ ]:
if result.chunks:
    chunk = result.chunks[0]
    print('chunk_id:', chunk.chunk_id)
    print('citation:', chunk.citation_anchor or chunk.citation_label)
    print('title:', chunk.title)
    print('scores:', {'vector': chunk.vector_score, 'rerank': chunk.rerank_score})
    print('--- text ---')
    print(chunk.chunk_text)
    print('--- metadata keys ---')
    print(sorted(chunk.metadata.keys()))
